# ✏️ PDF 필기 지우개

PDF를 올리면 필기를 지우고, **필기에 가려진 인쇄 글씨도 AI가 복원**합니다.

- 검정 펜 필기가 있는 흑백 스캔본 → **AI 모드** (기본)
- 색 펜·형광펜·연필, 태블릿 주석 → **빠른 모드**

메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 고르면 훨씬 빨라요. 셀을 위에서부터 차례로 ▶ 실행하세요.

In [ ]:
#@title 1. 설치
REPO = "https://github.com/hnr09/Clean_PDF"  #@param {type:"string"}
import os
if not os.path.exists('/content/Clean_PDF'):
    !git clone -q $REPO /content/Clean_PDF
%cd /content/Clean_PDF
!pip -q install pymupdf opencv-python-headless
!apt-get -qq install -y fonts-nanum fonts-nanum-extra > /dev/null  # 학습할 때만 필요

In [ ]:
#@title 2. PDF 업로드
from google.colab import files
up = files.upload()
in_path = list(up.keys())[0]
print('업로드:', in_path)

In [ ]:
#@title 3. (선택) 모델 추가 학습 — 결과가 아쉬울 때
#@markdown **필기 있는 내 스캔본 자체**로 미세조정하면 내 스캐너·시험지 양식에 맞춰져 가장 효과가 좋아요 (기존 필기는 자동으로 학습에서 제외). 필기 없는 원본 PDF가 있으면 같이 올리면 더 좋아요. GPU에서 10~20분.
학습하기 = False  #@param {type:"boolean"}
필기없는_원본PDF도_올리기 = False  #@param {type:"boolean"}
학습_스텝 = 3000  #@param {type:"integer"}
if 학습하기:
    clean = list(files.upload().keys()) if 필기없는_원본PDF도_올리기 else []
    cargs = ('--clean-pdf ' + ' '.join(f'"{c}"' for c in clean)) if clean else ''
    !python train.py --resume eraser.pt --scan-pdf "$in_path" $cargs --steps $학습_스텝 --batch 24 --size 256 --width 16 --workers 2 --lr 7e-4 --out eraser.pt

In [ ]:
#@title 4. 필기 지우기 실행
모드 = "AI (검정펜+복원)"  #@param ["AI (검정펜+복원)", "빠른 모드 (색펜/연필/태블릿)"]
DPI = 200  #@param {type:"slider", min:150, max:300, step:50}
민감도 = 0.5  #@param {type:"slider", min:0.2, max:0.8, step:0.05}
#@markdown 민감도: 낮출수록 더 많이 지움 (필기가 남으면 낮추고, 인쇄 글씨가 지워지면 높이세요)
out_path = in_path.rsplit('.', 1)[0] + '_clean.pdf'
if 모드.startswith('AI'):
    !python erase_ai.py "$in_path" -o "$out_path" --dpi $DPI --thr $민감도
else:
    !python erase_handwriting.py "$in_path" -o "$out_path" --mode auto --auto-pencil --dpi $DPI

In [ ]:
#@title 5. 전/후 비교 (첫 3쪽)
import pymupdf, matplotlib.pyplot as plt, numpy as np
a, b = pymupdf.open(in_path), pymupdf.open(out_path)
for i in range(min(3, len(a))):
    fig, ax = plt.subplots(1, 2, figsize=(16, 11))
    for k, (d, t) in enumerate([(a, 'before'), (b, 'after')]):
        px = d[i].get_pixmap(dpi=90, colorspace=pymupdf.csRGB)
        ax[k].imshow(np.frombuffer(px.samples, np.uint8).reshape(px.height, px.width, 3)); ax[k].set_title(f'p{i+1} {t}'); ax[k].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
#@title 6. 다운로드
files.download(out_path)

In [ ]:
#@title 7. (선택) 학습한 모델 내려받기
#@markdown 3번에서 추가 학습했다면 eraser.pt를 받아 GitHub 저장소의 eraser.pt를 교체하세요. 다음부터 바로 좋아진 모델을 써요.
files.download('eraser.pt')

In [ ]:
#@title 8. (관리자) Drive에 모인 학습 자료로 모델 개선
#@markdown 웹 페이지에서 사용자들이 보낸 자료(`내 드라이브/CleanPDF_학습자료`)로 모델을 미세조정하고, 새 모델 파일을 내려받아요. GPU 런타임 권장.
학습_스텝 = 4000  #@param {type:"integer"}
from google.colab import drive, files
drive.mount('/content/drive')
!pip -q install onnx onnxscript
!python train.py --resume eraser.pt --contrib-dir "/content/drive/MyDrive/CleanPDF_학습자료" --steps $학습_스텝 --batch 24 --size 256 --workers 2 --lr 5e-4 --out eraser.pt
!python export_onnx.py
files.download('eraser.pt'); files.download('docs/eraser.onnx')
print('저장소에 eraser.pt(루트)와 eraser.onnx(docs 폴더)를 덮어쓰면 웹 페이지가 새 모델을 사용해요.')